<!--nav--> [🗺 Learning path](README.md) · **31/43** · ◀ [Structured Output & Guided Decoding](./Structured_Output_Guided_Decoding.ipynb) · [Serving LoRA Adapters at Scale](./MultiLoRA_Serving_At_Scale.ipynb) ▶

# Distributed Serving: Tensor Parallelism, Replicas & Smart Routing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Distributed_MultiReplica_Serving.ipynb)

One GPU is never the final answer. Either the model doesn't fit, or the traffic doesn't fit. The
moment you add a second GPU you face a fork in the road that most teams get wrong:

> **Do I split the model across GPUs (tensor parallelism), or run a copy on each (replicas)?**

They optimize *different things*, and the answer changes with your SLO. Then, once you have
replicas, a second question appears that is worth more than most kernel optimizations:
**which replica should this request go to?**

| Part | What you'll learn |
|---|---|
| **1** | The four parallelism axes, and which problem each one solves |
| **2** | **TP vs replicas**, modeled and plotted: latency wins vs throughput wins |
| **3** | The **communication tax** — why TP needs NVLink, with the all-reduce math |
| **4** | **Routing**, simulated: round-robin vs least-loaded vs **prefix-aware** (the big one) |
| **5** | **Prefill/decode disaggregation** — the frontier architecture, and when it pays |
| **6** | Running it: vLLM flags, health checks, and a rollout checklist |

**Runs on:** any CPU (everything is modeled/simulated). Multi-GPU flags are shown for when you have
the hardware.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Four axes, four different problems

| Axis | What it splits | Solves | Costs |
|---|---|---|---|
| **Data parallel / replicas** (`DP`) | nothing — full copy per GPU | **throughput**, availability | full model memory per GPU |
| **Tensor parallel** (`TP`) | each layer's matrices, across GPUs | **model doesn't fit**; lowers latency | an all-reduce **per layer**, per token |
| **Pipeline parallel** (`PP`) | layers into stages | model doesn't fit; cheap interconnect OK | bubbles; higher latency |
| **Expert parallel** (`EP`) | MoE experts across GPUs | huge MoE models (DeepSeek, Mixtral) | all-to-all routing traffic |

The decision rule, in order:

1. **Does the model + KV fit on one GPU?** (weights + your KV pool from notebook 26's startup log.)
   If yes → **replicas**. Always. It's linear, simple, and failure-isolated.
2. **If it doesn't fit** → TP up to the smallest size that fits, *within one NVLink-connected node*.
3. **Still doesn't fit** (or crossing nodes) → add PP across nodes, TP inside them.
4. **MoE model** → EP, usually combined with TP.

Then replicate whatever unit you ended up with.

## Part 2 · TP vs replicas, modeled

Two GPUs, two strategies:

- **2 replicas**: two independent engines. Each request runs on one GPU at 1× speed. Aggregate
  capacity ≈ 2×. **Latency per request: unchanged.**
- **TP=2**: one engine spanning both GPUs. Each request's matmuls are split in half → faster per
  token — but every layer pays an **all-reduce** to recombine partial results.

TP speedup is Amdahl-shaped: compute halves, communication doesn't.

$$\text{speedup}_{TP=N} = \frac{1}{\frac{1}{N} + \frac{c \cdot (N-1)}{N}}$$

where `c` is communication cost as a fraction of one GPU's layer compute time.

In [ ]:
def tp_speedup(n, comm_frac):
    '''Amdahl-style TP model: compute divides by n; all-reduce grows with (n-1)/n.'''
    return 1.0 / (1.0 / n + comm_frac * (n - 1) / n)

INTERCONNECTS = {
    "NVLink (900 GB/s, H100)": 0.03,   # all-reduce is cheap
    "NVLink (300 GB/s, A100)": 0.06,
    "PCIe 4.0 x16 (32 GB/s)":  0.22,   # this is where TP starts hurting
    "Ethernet 25Gb (cross-node)": 0.75,
}

print(f"{'interconnect':<28}" + "".join(f"TP={n:<7}" for n in (2, 4, 8)))
print("-" * 60)
tp_rows = []
for name, c in INTERCONNECTS.items():
    sp = [tp_speedup(n, c) for n in (2, 4, 8)]
    tp_rows.append({"link": name, "comm": c,
                    **{f"tp{n}": tp_speedup(n, c) for n in (2, 4, 8)}})
    print(f"{name:<28}" + "".join(f"{s:<10.2f}" for s in sp))

print("\nRead the bottom row: over Ethernet, TP=8 is SLOWER than one GPU (0.62x).")
print("Tensor parallelism is a within-node technique. Cross-node, use pipeline parallel or replicas.")
print("\nEfficiency (speedup / GPUs used) - what you're actually paying for:")
for r in tp_rows:
    print(f"  {r['link']:<28} TP=2 {r['tp2']/2:.0%}  TP=4 {r['tp4']/4:.0%}  TP=8 {r['tp8']/8:.0%}")

In [ ]:
# Throughput vs latency: the actual decision. 8 GPUs, split between TP width and replica count.
COMM = 0.06                      # A100-class NVLink
BASE_TPS_PER_REPLICA = 3100      # single-GPU output tokens/s (notebook 27's A100 row)
BASE_TPOT_MS = 22.0              # per-token latency on one GPU

configs = []
for tp in (1, 2, 4, 8):
    replicas = 8 // tp
    sp = tp_speedup(tp, COMM)
    tpot = BASE_TPOT_MS / sp                       # TP makes each token faster
    # aggregate throughput: each replica's throughput scales with its TP speedup,
    # but you have fewer replicas
    total_tps = replicas * BASE_TPS_PER_REPLICA * sp
    configs.append({"tp": tp, "replicas": replicas, "speedup": sp,
                    "tpot_ms": tpot, "total_tps": total_tps,
                    "efficiency": sp / tp})

print(f"{'config':<18}{'TP speedup':>12}{'TPOT':>10}{'aggregate tok/s':>18}{'GPU efficiency':>16}")
print("-" * 74)
for c in configs:
    print(f"TP={c['tp']} x {c['replicas']} replicas{'':<3}{c['speedup']:>12.2f}"
          f"{c['tpot_ms']:>9.1f}ms{c['total_tps']:>18,.0f}{c['efficiency']:>15.0%}")

print("\nThe trade-off in one sentence:")
print("  TP buys LATENCY (22ms -> 8ms per token) and pays with THROUGHPUT (24.8k -> 11.3k tok/s).")
print("  Replicas buy THROUGHPUT and leave latency alone.")
print("\nSo: TP only as wide as you must (to fit the model, or to hit a hard TPOT SLO). Then replicate.")

In [ ]:
# Interactive: how wide should TP be, given YOUR latency SLO?
JS = r'''
const M = {top: 20, right: 64, bottom: 46, left: 62};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const ctr = root.append("div").style("font-size","13px").style("margin-bottom","6px");
const w = ctr.append("label"); w.append("span").text("TPOT SLO: ");
const out = w.append("b").text("15.0ms");
const inp = w.append("input").attr("type","range").attr("min",5).attr("max",25).attr("step",0.5)
    .attr("value",15).style("vertical-align","middle").style("margin-left","6px");
const verdict = root.append("div").style("font","13px system-ui").style("margin-bottom","6px");

const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleBand().domain(data.map(d=>`TP=${d.tp}×${d.replicas}`)).range([0,iw]).padding(0.3);
const y = d3.scaleLinear().domain([0, d3.max(data,d=>d.total_tps)*1.15]).range([ih,0]);
const y2 = d3.scaleLinear().domain([0, d3.max(data,d=>d.tpot_ms)*1.2]).range([ih,0]);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x));
svg.append("g").call(d3.axisLeft(y).ticks(5,"~s"));
svg.append("g").attr("transform",`translate(${iw},0)`).call(d3.axisRight(y2).ticks(5));
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-46)
   .attr("text-anchor","middle").style("font-size","12px").style("fill","#1976d2")
   .text("aggregate tokens/s");
svg.append("text").attr("transform","rotate(90)").attr("x",ih/2).attr("y",-iw-46)
   .attr("text-anchor","middle").style("font-size","12px").style("fill","#d32f2f").text("TPOT (ms)");

const bars = svg.selectAll("b").data(data).join("rect")
    .attr("x",d=>x(`TP=${d.tp}×${d.replicas}`)).attr("width",x.bandwidth())
    .attr("y",d=>y(d.total_tps)).attr("height",d=>ih-y(d.total_tps)).attr("rx",3);
svg.append("path").datum(data).attr("fill","none").attr("stroke","#d32f2f").attr("stroke-width",2.5)
   .attr("d", d3.line().x(d=>x(`TP=${d.tp}×${d.replicas}`)+x.bandwidth()/2).y(d=>y2(d.tpot_ms)));
svg.selectAll("dot").data(data).join("circle")
   .attr("cx",d=>x(`TP=${d.tp}×${d.replicas}`)+x.bandwidth()/2).attr("cy",d=>y2(d.tpot_ms))
   .attr("r",4).attr("fill","#d32f2f");
const sloLine = svg.append("line").attr("x1",0).attr("x2",iw).attr("stroke","#d32f2f")
   .attr("stroke-dasharray","4 3");

function draw() {
  const slo = +inp.node().value; out.text(slo.toFixed(1)+"ms");
  sloLine.attr("y1",y2(slo)).attr("y2",y2(slo));
  const feasible = data.filter(d => d.tpot_ms <= slo);
  const best = feasible.length ? feasible.reduce((a,b)=>a.total_tps>b.total_tps?a:b) : null;
  bars.attr("fill", d => d.tpot_ms > slo ? "#cfd8dc" : (best && d.tp===best.tp ? "#2e7d32" : "#90caf9"));
  verdict.html(best
    ? `✅ best config meeting a ${slo.toFixed(1)}ms TPOT SLO: <b>TP=${best.tp} × ${best.replicas} replicas</b> ` +
      `→ ${d3.format(",.0f")(best.total_tps)} tok/s aggregate (grey bars miss the SLO)`
    : `❌ no configuration on 8 GPUs meets a ${slo.toFixed(1)}ms TPOT — you need faster GPUs, ` +
      `quantization (nb 23), or speculative decoding (nb 24)`);
}
inp.on("input", draw); draw();
'''
show_d3(JS, configs, height=360)

**Drag the SLO slider.** With a relaxed SLO (20ms+) the winner is always `TP=1 × 8 replicas` —
maximum throughput. Tighten it and you're forced to spend GPUs on TP width instead, buying latency
with throughput you no longer get to sell. **Your latency promise has a direct GPU cost**, and this
chart is how you price it.

## Part 3 · The communication tax, concretely

Every TP layer does an **all-reduce** over the hidden state. Per token, per layer:

```
bytes moved ≈ 2 (bf16) × hidden_size × batch × 2(N−1)/N        [ring all-reduce]
```

For Llama-3-70B (hidden 8192, 80 layers) at TP=4, batch 32:

- per layer: `2 × 8192 × 32 × 1.5 ≈ 786 KB`
- per token, all layers: `× 80 ≈ 63 MB`
- at 20 tokens/s/request... you are moving **gigabytes per second** of pure coordination traffic.

On NVLink (900 GB/s) that's a rounding error. On PCIe (32 GB/s, shared) it's your bottleneck. This
is why "TP=8 across two 4-GPU nodes over Ethernet" performs worse than a single GPU — the table in
Part 2 wasn't hypothetical.

**Practical rules:**
- Keep `--tensor-parallel-size` **≤ the number of NVLink-connected GPUs in one node**.
- Cross-node → `--pipeline-parallel-size` (stage boundaries move activations once per stage, not
  per layer).
- Check your topology before you trust any TP benchmark: `nvidia-smi topo -m` (look for `NV#` links
  vs `SYS`/`PHB`).

## Part 4 · Routing: the cheapest big win you're not using

With N replicas, a load balancer picks one per request. The default is round-robin. That is
**leaving a 2–4× on the table** for any workload with shared prefixes — because notebook 22's prefix
cache is *per replica*. Send a conversation's turn 2 to a different replica than turn 1, and you
re-prefill the entire history from scratch.

Let's simulate three routers on realistic multi-turn traffic:

In [ ]:
# Multi-tenant chat traffic: sessions with growing histories + shared system prompts per tenant.
random.seed(11)
N_REPLICAS, N_SESSIONS, N_TENANTS = 4, 60, 6
CACHE_CAPACITY = 40                     # distinct prefixes a replica keeps hot (block-pool proxy)

def make_requests(n=600):
    reqs = []
    for i in range(n):
        sess = random.randrange(N_SESSIONS)
        tenant = sess % N_TENANTS
        turn = sum(1 for r in reqs if r["session"] == sess)
        # prefix key = tenant system prompt + this session's history so far
        reqs.append({"i": i, "session": sess, "tenant": tenant, "turn": turn,
                     "prefix": f"t{tenant}/s{sess}/turn{turn}",
                     "prefix_tokens": 400 + 350 * turn,      # history grows every turn
                     "out_tokens": random.randint(60, 240)})
    return reqs

REQS = make_requests()

def route_simulate(policy):
    caches = [dict() for _ in range(N_REPLICAS)]      # replica -> {prefix_key: recency}
    load = [0] * N_REPLICAS                            # outstanding work per replica
    hits = prefill_tokens = 0
    clock = 0
    for r in REQS:
        clock += 1
        if policy == "round_robin":
            rep = r["i"] % N_REPLICAS
        elif policy == "least_loaded":
            rep = min(range(N_REPLICAS), key=lambda k: load[k])
        elif policy == "session_affinity":
            rep = r["session"] % N_REPLICAS
        elif policy == "prefix_aware":
            # prefer a replica that already holds this prefix; break ties by load
            holders = [k for k in range(N_REPLICAS) if r["prefix"] in caches[k]]
            if holders:
                rep = min(holders, key=lambda k: load[k])
            else:
                # fall back to least-loaded, but keep sessions sticky to preserve future hits
                rep = min(range(N_REPLICAS), key=lambda k: (load[k], k))
        hit = r["prefix"] in caches[rep]
        hits += hit
        prefill_tokens += 0 if hit else r["prefix_tokens"]
        # this request's own prefix (history including this turn) becomes cacheable
        caches[rep][r["prefix"]] = clock
        caches[rep][f"t{r['tenant']}/s{r['session']}/turn{r['turn']+1}"] = clock   # next turn's prefix
        if len(caches[rep]) > CACHE_CAPACITY:                                       # LRU eviction
            for k in sorted(caches[rep], key=caches[rep].get)[:len(caches[rep]) - CACHE_CAPACITY]:
                del caches[rep][k]
        load[rep] += r["out_tokens"]
    spread = (max(load) - min(load)) / max(statistics.mean(load), 1)
    return {"policy": policy, "hit_rate": hits / len(REQS),
            "prefill_tokens": prefill_tokens, "imbalance": spread}

results = [route_simulate(p) for p in
           ("round_robin", "least_loaded", "session_affinity", "prefix_aware")]
base = next(r for r in results if r["policy"] == "round_robin")
print(f"{'router':<20}{'prefix hit rate':>17}{'prefill tokens':>17}{'saved':>9}{'load imbalance':>16}")
print("-" * 80)
for r in results:
    saved = 1 - r["prefill_tokens"] / base["prefill_tokens"]
    print(f"{r['policy']:<20}{r['hit_rate']:>17.1%}{r['prefill_tokens']:>17,}"
          f"{saved:>8.0%}{r['imbalance']:>15.0%}")

print("\nSame GPUs, same requests, same engine - only the load balancer changed.")
print("Prefix-aware routing is a config change in your gateway that can rival a hardware upgrade.")

In [ ]:
# Visualize where requests land and how the cache heats up, per policy.
viz = {"policies": [r["policy"] for r in results],
       "hit_rate": [r["hit_rate"] for r in results],
       "prefill": [r["prefill_tokens"] for r in results],
       "imbalance": [r["imbalance"] for r in results]}

JS = r'''
const M = {top: 24, right: 24, bottom: 60, left: 70};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleBand().domain(data.policies).range([0,iw]).padding(0.22);
const y = d3.scaleLinear().domain([0,1]).range([ih,0]);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x))
   .selectAll("text").attr("transform","rotate(-12)").style("text-anchor","end");
svg.append("g").call(d3.axisLeft(y).tickFormat(d3.format(".0%")));
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-48)
   .attr("text-anchor","middle").style("font-size","12px").text("prefix cache hit rate");

const maxPrefill = d3.max(data.prefill);
svg.selectAll("bar").data(data.policies).join("rect")
   .attr("x",(d,i)=>x(d)).attr("width",x.bandwidth())
   .attr("y",(d,i)=>y(data.hit_rate[i])).attr("height",(d,i)=>ih-y(data.hit_rate[i]))
   .attr("rx",4).attr("fill",(d,i)=> d==="prefix_aware" ? "#2e7d32" : "#90a4ae");
svg.selectAll("lab").data(data.policies).join("text")
   .attr("x",(d,i)=>x(d)+x.bandwidth()/2).attr("y",(d,i)=>y(data.hit_rate[i])-16)
   .attr("text-anchor","middle").style("font-size","11.5px").style("font-weight",600)
   .text((d,i)=>d3.format(".0%")(data.hit_rate[i]));
svg.selectAll("lab2").data(data.policies).join("text")
   .attr("x",(d,i)=>x(d)+x.bandwidth()/2).attr("y",(d,i)=>y(data.hit_rate[i])-3)
   .attr("text-anchor","middle").style("font-size","10.5px").style("fill","#546e7a")
   .text((d,i)=>`${d3.format(".2s")(data.prefill[i])} prefill tok`);
'''
show_d3(JS, viz, height=330)

**In production this is what "KV-aware routing" means** — and it's why the routing layer has
become a real piece of infrastructure:

- **NVIDIA Dynamo** and **vLLM's production stack** ship KV-aware routers that track which replica
  holds which blocks and route accordingly.
- **Mooncake** (which serves Kimi) goes further: a *global* KV store, so any replica can pull a
  prefix another one computed.
- The cheap 80% version, available to everyone today: **session affinity** — hash the conversation
  ID to a replica. Note in the table above how close it gets to full prefix-awareness for chat.

**Warning:** affinity fights load balance (see the imbalance column). Good routers blend both —
"prefer the cache holder *unless* it's much busier than the alternatives."

## Part 5 · Prefill/decode disaggregation

Notebook 21 established that prefill is **compute-bound** and decode is **memory-bandwidth-bound**.
Running both on the same GPU means one always compromises the other: a long prefill stalls everyone's
token stream (chunked prefill mitigates, doesn't eliminate).

Disaggregation runs them on **separate GPU pools**:

```
            ┌──────────────┐   KV cache transfer   ┌─────────────┐
 request ─► │ PREFILL pool │ ────────────────────► │ DECODE pool │ ─► tokens ─► user
            │ compute-rich │   (NVLink/RDMA)       │ bandwidth-  │
            │ big batches  │                       │ rich, many  │
            └──────────────┘                       │ concurrent  │
                                                   └─────────────┘
```

**Wins:** each pool is tuned and scaled independently (prefill-heavy RAG? add prefill nodes); no
interference; per-phase batching that's actually optimal.
**Costs:** you must ship the KV cache between pools — that's the whole prompt's cache, once per
request. Worth it when prompts are long and the interconnect is fast; a loss otherwise.

**Rule of thumb:** disaggregation pays off at scale (many nodes) with long prompts. Below ~8 GPUs,
chunked prefill on colocated pools is simpler and usually as good.

In [ ]:
# When does disaggregation pay? Compare colocated (with interference) vs disaggregated (with transfer).
def compare(prompt_len, out_len=200, kv_kb_per_token=128, link_gbs=50,
            prefill_tps=9000, decode_tps=25, interference=0.30):
    '''interference = fraction of decode throughput lost to prefill sharing the GPU (colocated).'''
    colocated = prompt_len / prefill_tps + out_len / (decode_tps * (1 - interference))
    transfer_s = (prompt_len * kv_kb_per_token * 1024) / (link_gbs * 1e9)
    disagg = prompt_len / prefill_tps + transfer_s + out_len / decode_tps
    return colocated, disagg, transfer_s

print(f"{'prompt tokens':>14}{'colocated':>12}{'disaggregated':>15}{'KV transfer':>13}{'winner':>16}")
print("-" * 72)
for pl in (256, 1024, 4096, 16384, 65536):
    c, d, t = compare(pl)
    print(f"{pl:>14,}{c:>11.2f}s{d:>14.2f}s{t*1000:>11.0f}ms"
          f"{('disaggregated' if d < c else 'colocated'):>16}")

print("\nShort prompts: the KV transfer is pure overhead - stay colocated.")
print("Long prompts: prefill interference dominates, and disaggregation wins.")
print("The crossover moves LEFT with a faster interconnect and RIGHT with a slower one -")
print("which is why this is a datacenter-scale technique, not a two-GPU one.")

## Part 6 · Running it

**vLLM flags:**

```bash
# Tensor parallel across 4 NVLink-connected GPUs in one node
vllm serve meta-llama/Llama-3.1-70B-Instruct \
  --tensor-parallel-size 4 \
  --max-model-len 8192 --gpu-memory-utilization 0.90

# TP inside nodes, PP across them (2 nodes × 4 GPUs)
vllm serve <model> --tensor-parallel-size 4 --pipeline-parallel-size 2

# MoE with expert parallelism
vllm serve <moe-model> --tensor-parallel-size 8 --enable-expert-parallel
```

**Replicas** are just N independent servers behind your load balancer. Give each a health check on
`/health`, scrape `/metrics` per replica (notebook 26), and **never** let the LB send traffic to a
replica that's still doing engine init — startup can take a minute (that whole startup log!).

**Rollout checklist:**

- [ ] `nvidia-smi topo -m` — confirm TP GPUs are NVLink-connected (`NV#`, not `SYS`)
- [ ] TP width is the **smallest** that fits the model + your KV pool
- [ ] Per-replica `/health` gating in the load balancer; drain before shutdown
- [ ] Router does **session affinity at minimum**; prefix-aware if your gateway supports it
- [ ] Capacity math from notebook 27 done **per replica**, then multiplied
- [ ] Headroom for **N−1 replicas** — one node dies, the rest must absorb it without breaching SLO
- [ ] Metrics labeled by replica so one sick GPU doesn't hide in the average

## Recap

1. **Replicas for throughput, TP for fit and latency.** Never TP wider than you must.
2. **TP is bounded by your interconnect** — an all-reduce per layer per token. NVLink or don't bother.
3. **Routing is a first-class optimization.** Prefix-aware/session-affinity routing preserves the
   per-replica prefix cache and can save a large share of all prefill work — measured in Part 4.
4. **Disaggregation** separates two workloads that never wanted to share a GPU; it pays off with long
   prompts, at scale, over fast links.
5. Plan capacity for **N−1**, not N.

### Further reading
- [Megatron-LM](https://arxiv.org/abs/1909.08053) — the original tensor-parallel formulation
- [vLLM distributed serving docs](https://docs.vllm.ai/en/latest/serving/distributed_serving.html)
- [DistServe](https://arxiv.org/abs/2401.09670) · [Mooncake](https://arxiv.org/abs/2407.00079) — disaggregation & global KV
- [NVIDIA Dynamo](https://github.com/ai-dynamo/dynamo) — KV-aware routing in a production router

▶ **Next:** [Serving LoRA Adapters at Scale](./MultiLoRA_Serving_At_Scale.ipynb) — one base model,
fifty fine-tunes, one GPU.